# 第 19 章互动实验：迷你 PatchCore

先用简单的局部纹理特征把你实现的零件串成完整流程，不需要下载任何权重；最后一节可选地换成预训练 ResNet 特征。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "19_异常检测与深度缺陷检测"
ASSIGNMENT = "19-1_PatchCore核心.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
def make_image(seed, defect=False):
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:64, 0:64]
    img = 0.5 + 0.3 * np.sin(2 * np.pi * x / 8 + rng.uniform(0, 0.3)) + rng.normal(scale=0.06, size=(64, 64))
    mask = np.zeros((64, 64), bool)
    if defect:
        cy, cx = rng.integers(10, 54, 2); r = rng.integers(2, 4)
        mask = (y - cy) ** 2 + (x - cx) ** 2 <= r * r
        img[mask] += rng.choice([-0.2, 0.2])
    return img, mask

def patch_features(img, p=4):
    """把每个 4x4 patch 展平成 16 维向量，再拼上邻域平均（模仿 PatchCore 的局部聚合）。"""
    h, w = img.shape
    pats = np.lib.stride_tricks.sliding_window_view(img, (p, p))[::2, ::2]
    feats = pats.reshape(pats.shape[0], pats.shape[1], -1)
    pad = np.pad(feats, ((1, 1), (1, 1), (0, 0)), mode="edge")
    agg = sum(pad[dy:dy + feats.shape[0], dx:dx + feats.shape[1]] for dy in range(3) for dx in range(3)) / 9
    return agg

train = [make_image(s)[0] for s in range(20)]
bank = np.concatenate([patch_features(im).reshape(-1, 16) for im in train])
print("记忆库大小：", bank.shape)

In [ ]:
idx = attempt(mod.greedy_coreset, bank, max(1, len(bank) // 20))
if idx is not None:
    memory = bank[idx]
    print("核心集大小：", memory.shape)
    test = [make_image(1000 + s, defect=(s % 2 == 0)) for s in range(40)]
    labels = np.array([int(m.any()) for _, m in test])
    img_scores, maps = [], []
    for im, _ in test:
        f = patch_features(im); s = mod.knn_scores(memory, f.reshape(-1, 16), k=1).reshape(f.shape[:2])
        maps.append(s); img_scores.append(s.max())
    img_scores = np.array(img_scores)
    print("Image AUROC：", round(mod.auroc(img_scores, labels), 4))
    fig, ax = plt.subplots(2, 4, figsize=(12, 6))
    for i in range(4):
        ax[0, i].imshow(test[i][0], cmap="gray"); ax[0, i].contour(test[i][1], colors="c")
        ax[1, i].imshow(maps[i], cmap="magma"); ax[1, i].set_title(f"分数 {img_scores[i]:.2f}")
    plt.show()

## 实验：从 AUROC 到产线工作点

要求漏检率不超过 0%、5%、10% 时，过检率分别是多少？再看核心集比例从 100% 压到 1%，AUROC 掉了多少，想想为什么掉得这么少。

In [ ]:
if idx is not None:
    for esc in (0.0, 0.05, 0.10):
        r = attempt(mod.threshold_at_escape, img_scores[labels == 0], img_scores[labels == 1], esc)
        if r: print(f"漏检上限 {esc:.0%}：阈值 {r[0]:.3f}，实际漏检 {r[1]:.0%}，过检 {r[2]:.0%}")
    for frac in (1.0, 0.2, 0.05, 0.01):
        m = max(1, int(len(bank) * frac))
        mem = bank[mod.greedy_coreset(bank, m)] if frac < 1 else bank
        sc = [mod.knn_scores(mem, patch_features(im).reshape(-1, 16)).max() for im, _ in test]
        print(f"核心集比例 {frac:>5.0%}：AUROC {mod.auroc(np.array(sc), labels):.4f}")

## 选做：换成预训练 ResNet 特征

需要 torch、torchvision 和网络（首次下载权重）。在 MVTec AD 上跑时，建议直接用 Anomalib，把它的结果和这里的迷你版对照。

In [ ]:
try:
    import torch, torchvision
    weights = torchvision.models.ResNet18_Weights.DEFAULT
    net = torchvision.models.resnet18(weights=weights).eval()
    print("已加载 ResNet18。取 layer2 和 layer3 的输出、上采样拼接，就得到 PatchCore 的 patch 特征。")
except Exception as exc:
    print("跳过：", type(exc).__name__, exc)

## 破坏实验与记录

- 训练集里混入 2 张带缺陷的图，AUROC 怎么变？为什么？
- 去掉局部聚合（直接用 16 维 patch），结果怎么变？
- 把漏检约束下的过检率写进 `实验记录.md`。